# VDAnta Statistics 

In [1]:
# The data has the following structure:
# 1. Name of the trainer 
# 2. Results of the year 2020/2021
# 3. Results of the year 2021/2022
# 4. Results of the year 2022/2023

# The results are:
# W = Winner
# Q = Qualified
# R = Relegated
# N = Not qualified
# A = Absent

original_data =[['Lippo',   'W', 'Q', 'R', 'N', 'N'], 
                ['Gallo',   'Q', 'R', 'R', 'R', 'W'], 
                ['Cappo',   'N', 'N', 'W', 'R', 'Q'], 
                ['Cucu',    'N', 'Q', 'Q', 'Q', 'N'], 
                ['Manu',    'Q', 'W', 'N', 'N', 'Q'], 
                ['Mett',    'R', 'R', 'A', 'A', 'A'],
                ['Fabio',   'R', 'R', 'N', 'N', 'R'], 
                ['Zanu',    'N', 'N', 'R', 'N', 'N'], 
                ['Pippi',   'R', 'N', 'Q', 'W', 'R'], 
                ['Furlan',  'A', 'A', 'N', 'A', 'N'], 
                ['Alex',    'A', 'A', 'N', 'A', 'A'], 
                ['Jackino', 'A', 'A', 'A', 'R', 'A'], 
                ['Javier',  'A', 'A', 'A', 'Q', 'A'], 
                ['Luca',    'A', 'A', 'A', 'A', 'R'], 
                ]


## Statistics 

### Historic ranking
For all palyer, for all year, the sum of the results of all the past years, where 
- win 
- podium 
- partecipations 
- relagations 
  
Then, the order is done depending on who have more win. In case of same number of win on who has more podium, in case of same number on who has more partecipation and then on who have less relagations. 

### Ranking 
For all palyer, for all year, the sum of the results of all the past years, where 
- win = 10
- podium = 8
- middle = 6
- absent = 5.75
- relegated = 4

If $r_1,r_2,r_3$ are the results of the last 3 years, then the formula that gives the rank is 
$$
\text{rank} = (\alpha_1 r_1 + \alpha_2 r_2 + \alpha_3 r_3) \times 100
$$
where $\alpha_1 = 0.2, \alpha_2 = 0.3, \alpha_3 = 0.5$. \
If, there were only 2 years then $\alpha_1 = 0.4, \alpha_2 = 0.6 $ \
If, there was only 1 year1 then $\alpha_1 = 1$.


## Historic ranking

In [2]:
# Libs 
import pandas as pd
from collections import Counter
from tabulate import tabulate

# Define the PlayerStats class
class PlayerStats:
    def __init__(self, player):
        self.name = player[0]
        self.wins = player.count("W")
        self.qualifications = player.count("W") + player.count("Q")
        self.participations = player.count("W") + player.count("Q") + player.count("N") + player.count("R")
        self.relegations = player.count("R")
    
    def to_dict(self):
        return {
            "Wins": self.wins,
            "Qualifications": self.qualifications,
            "Participations": self.participations,
            "Relegations": self.relegations
        }
    
# Define the compute_historical_ranking function
def compute_historical_ranking(players, num_seasons):

    # Select the number of seasons from the beginning 
    corrected_players = []
    for player in players:
        corrected_players.append(player[0:num_seasons+1])

    # Create PlayerStats objects
    player_stats_list = [PlayerStats(player) for player in corrected_players]

    # Create a DataFrame
    data = {player_stats.name: player_stats.to_dict() for player_stats in player_stats_list}
    df = pd.DataFrame(data).T  # Transpose to have players as rows

    # Sort the DataFrame based on the criteria
    df = df.sort_values(by=["Wins", "Qualifications", "Participations", "Relegations"], 
                    ascending=[False, False, False, True])

    # Add index
    df = df.reset_index()
    df.index += 1
    
    # Rename index clumns as Name 
    df.rename(columns={"index": "Name"}, inplace=True)


    return df

# Example of usage 
players = [
    ["Player1", "W", "W", "Q", "R", "N", "N"],
    ["Player2", "W", "A", "W", "Q", "Q", "N"],
    ["Player3", "W", "W", "Q", "A", "Q", "R"]
]
num_seasons = 2

table = compute_historical_ranking(players = players, num_seasons = num_seasons)
table 


,Name,Wins,Qualifications,Participations,Relegations
1,Player1,2,2,2,0
2,Player3,2,2,2,0
3,Player2,1,1,1,0


Application

In [12]:
a = 2023 # first year of the season 
first_year_ever = 2020

# Automatic 
b = a + 1
num_seasons = a - first_year_ever + 1
print(f"Season: {a}/{b}")
table = compute_historical_ranking(players = original_data, num_seasons = num_seasons)
table = tabulate(table, headers='keys', tablefmt='fancy_grid')
print(table)

Season: 2023/2024
╒════╤═════════╤════════╤══════════════════╤══════════════════╤═══════════════╕
│    │ Name    │   Wins │   Qualifications │   Participations │   Relegations │
╞════╪═════════╪════════╪══════════════════╪══════════════════╪═══════════════╡
│  1 │ Manu    │      1 │                2 │                4 │             0 │
├────┼─────────┼────────┼──────────────────┼──────────────────┼───────────────┤
│  2 │ Lippo   │      1 │                2 │                4 │             1 │
├────┼─────────┼────────┼──────────────────┼──────────────────┼───────────────┤
│  3 │ Pippi   │      1 │                2 │                4 │             1 │
├────┼─────────┼────────┼──────────────────┼──────────────────┼───────────────┤
│  4 │ Cappo   │      1 │                1 │                4 │             1 │
├────┼─────────┼────────┼──────────────────┼──────────────────┼───────────────┤
│  5 │ Cucu    │      0 │                3 │                4 │             0 │
├────┼─────────┼──────

## Current ranking 

In [4]:
def translate_player(player):
    # Define the translation dictionary
    translation = {
        'W': 10,
        'Q': 8,
        'N': 6,
        'A': 5.75,
        'R': 4
    }

    # Translate each element using the dictionary
    translated_player = [player[0]]  # Keep the name as the first element
    translated_player += [translation.get(item, item) for item in player[1:]]

    return translated_player


# Define the calculate_weighted_score function
# "how many years back" is a variable that allows to calculate the weighted score for a specific year:
# 0 is the current year, 1 is the previous year, 2 is the year before the previous year, etc.
def calculate_weighted_score(player, how_many_years_back = 0):
    # Extract the name
    name = player[0]
    
    # Calculate the weighted score
    weighted_score = round((0.5 * player[-1 - how_many_years_back] + 0.3 * player[-2 - how_many_years_back] + 0.2 * player[-3 - how_many_years_back])*10)
    
    return [name, weighted_score]

def calculate_weighted_score_2yearsAvailable(player, how_many_years_back = 0):
    # Extract the name
    name = player[0]
    
    # Calculate the weighted score
    weighted_score = round((0.65 * player[-1 - how_many_years_back] + 0.35 * player[-2 - how_many_years_back])*10)
    
    return [name, weighted_score]

def calculate_weighted_score_1yearAvailable(player, how_many_years_back = 0):
    # Extract the name
    name = player[0]
    
    # Calculate the weighted score
    weighted_score = round((1 * player[-1 - how_many_years_back])*10)
    
    return [name, weighted_score]


# Define the compute_current_ranking function
# "how many years back" is a variable that allows to calculate the weighted score for a specific year:
# 0 is the current year, 1 is the previous year, 2 is the year before the previous year, etc.
def compute_current_ranking(players, ranking_name="Ranking", how_many_years_back = 0, onlyTwoYearsAvailable = False, onlyOneYearAvailable = False):
    players_ranking = []
    
    for player in players:
        # Translate the player's array
        translated_player = translate_player(player)    
        
        if onlyOneYearAvailable:
            # Calculate the weighted score
            name, weighted_score = calculate_weighted_score_1yearAvailable(translated_player, how_many_years_back)
        elif onlyTwoYearsAvailable:
            # Calculate the weighted score
            name, weighted_score = calculate_weighted_score_2yearsAvailable(translated_player, how_many_years_back)
        else:
            # Calculate the weighted score
            name, weighted_score = calculate_weighted_score(translated_player, how_many_years_back)
        
        # Append the name and score as a tuple
        players_ranking.append((name, weighted_score))
    
    # Create a DataFrame
    df = pd.DataFrame(players_ranking, columns=['Name', ranking_name])
    
    # Sort the DataFrame by the Ranking column in descending order
    df_sorted = df.sort_values(by=ranking_name, ascending=False)
    df_sorted.reset_index(drop=True, inplace=True)
    df_sorted.index += 1
    
    return df_sorted

# Example usage
players = [
    ['Lippo', 'W', 'Q', 'R', 'N', 'A'],
    ['John', 'Q', 'W', 'N', 'N', 'A'],
    ['Mike', 'R', 'A', 'Q', 'W', 'N']
]

table = compute_current_ranking(players)
table = tabulate(table, headers='keys', tablefmt='fancy_grid')
print(table)


╒════╤════════╤═══════════╕
│    │ Name   │   Ranking │
╞════╪════════╪═══════════╡
│  1 │ Mike   │        76 │
├────┼────────┼───────────┤
│  2 │ John   │        59 │
├────┼────────┼───────────┤
│  3 │ Lippo  │        55 │
╘════╧════════╧═══════════╛


Application

In [14]:
ssn_year = 2024 # autumn season year you want the rank
current_year = 2025 # current year
first_year_ever = 2020 # first year of the fanta 

# automatic 
ssn_year_springer = ssn_year+1 
ranking_name = f'Ranking after {str(ssn_year)}/{str(ssn_year_springer)}'
how_many_years_back = current_year - ssn_year - 1
if ssn_year == first_year_ever:
    onlyOneYearAvailable = True 
    onlyTwoYearAvailable = False 
elif (ssn_year-1) == first_year_ever:
    onlyOneYearAvailable = False
    onlyTwoYearAvailable = True 
else:
    onlyOneYearAvailable = False
    onlyTwoYearAvailable = False  


table = compute_current_ranking(
    original_data,
    ranking_name=ranking_name,
    how_many_years_back=how_many_years_back, 
    onlyOneYearAvailable=onlyOneYearAvailable,
    onlyTwoYearsAvailable=onlyTwoYearAvailable
    )
table = tabulate(table, headers='keys', tablefmt='fancy_grid')
print(table)

╒════╤═════════╤═══════════════════════════╕
│    │ Name    │   Ranking after 2024/2025 │
╞════╪═════════╪═══════════════════════════╡
│  1 │ Cappo   │                        72 │
├────┼─────────┼───────────────────────────┤
│  2 │ Gallo   │                        70 │
├────┼─────────┼───────────────────────────┤
│  3 │ Manu    │                        70 │
├────┼─────────┼───────────────────────────┤
│  4 │ Cucu    │                        70 │
├────┼─────────┼───────────────────────────┤
│  5 │ Pippi   │                        66 │
├────┼─────────┼───────────────────────────┤
│  6 │ Javier  │                        64 │
├────┼─────────┼───────────────────────────┤
│  7 │ Furlan  │                        59 │
├────┼─────────┼───────────────────────────┤
│  8 │ Mett    │                        58 │
├────┼─────────┼───────────────────────────┤
│  9 │ Alex    │                        58 │
├────┼─────────┼───────────────────────────┤
│ 10 │ Lippo   │                        56 │
├────┼────